# 11.12 Forecast Evaluation Metrics

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.12_forecast-evaluation-metrics.ipynb)

Companion code for Section 11.12 of the book. Run the cells in order from top to bottom.

**Example 11.12.1** · bash

```bash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def forecast_metrics(y_true, y_pred, y_train=None, m=1,
                       y_lower=None, y_upper=None):
    '''Compute all forecast evaluation metrics.'''
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    errors = y_true - y_pred
    abs_errors = np.abs(errors)

    # MAE
    mae = abs_errors.mean()

    # RMSE
    rmse = np.sqrt((errors**2).mean())

    # MAPE (exclude zeros)
    mask = y_true != 0
    mape = np.mean(abs_errors[mask] / np.abs(y_true[mask])) * 100

    # sMAPE
    smape = np.mean(2*abs_errors / (np.abs(y_true)+np.abs(y_pred)+1e-10)) * 100

    # MASE
    if y_train is not None:
        naive_mae = np.mean(np.abs(np.diff(y_train, n=m)))
        mase = mae / naive_mae if naive_mae > 0 else np.nan
    else:
        mase = np.nan

    # RMSSE
    if y_train is not None:
        naive_mse = np.mean(np.diff(y_train, n=m)**2)
        rmsse = np.sqrt((errors**2).mean() / naive_mse) if naive_mse > 0 else np.nan
    else:
        rmsse = np.nan

    # Coverage
    if y_lower is not None and y_upper is not None:
        coverage = np.mean((y_true >= y_lower) & (y_true <= y_upper)) * 100
    else:
        coverage = np.nan

    # Bias
    bias = errors.mean()

    return {
        'MAE': mae, 'RMSE': rmse, 'MAPE%': mape,
        'sMAPE%': smape, 'MASE': mase, 'RMSSE': rmsse,
        'Coverage%': coverage, 'Bias': bias
    }


# ── Evaluate multiple models on Air Passengers ────────────────────
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

data = sm.datasets.get_rdataset('AirPassengers').data
ts = pd.Series(data['value'].values,
               index=pd.date_range('1949-01', periods=144, freq='MS'))

train = ts[:'1958-12'].values
test  = ts['1959-01':].values
h = len(test)

print('=== Forecast Evaluation - Air Passengers (24-month horizon) ===')
print(f'{'Method':<22} {'MAE':>6} {'RMSE':>7} {'MAPE%':>7} {'sMAPE%':>8} {'MASE':>6} {'Bias':>8}')
print('='*70)

# Naive: seasonal
pred_naive = np.array([train[-(12-i)%12-12] for i in range(h)])
m_naive = forecast_metrics(test, pred_naive, train, m=12)
print(f'{'Seasonal Naive':<22} {m_naive["MAE"]:>6.1f} {m_naive["RMSE"]:>7.1f} {m_naive["MAPE%"]:>7.2f}
{m_naive["sMAPE%"]:>8.2f} {m_naive["MASE"]:>6.3f} {m_naive["Bias"]:>8.2f}')

# Holt-Winters
hw = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=12)
hw_fit = hw.fit(optimized=True)
pred_hw = hw_fit.forecast(h)
m_hw = forecast_metrics(test, pred_hw, train, m=12)
print(f'{'Holt-Winters':<22} {m_hw["MAE"]:>6.1f} {m_hw["RMSE"]:>7.1f} {m_hw["MAPE%"]:>7.2f}
{m_hw["sMAPE%"]:>8.2f} {m_hw["MASE"]:>6.3f} {m_hw["Bias"]:>8.2f}')

# SARIMA (on log scale)
sarima = SARIMAX(np.log(train), order=(1,1,1), seasonal_order=(1,1,1,12),
                 enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = sarima.fit(disp=False)
pred_sarima = np.exp(sarima_fit.forecast(h))
m_sarima = forecast_metrics(test, pred_sarima, train, m=12)
print(f'{'SARIMA(1,1,1)(1,1,1)[12]':<22} {m_sarima["MAE"]:>6.1f} {m_sarima["RMSE"]:>7.1f}
{m_sarima["MAPE%"]:>7.2f} {m_sarima["sMAPE%"]:>8.2f} {m_sarima["MASE"]:>6.3f} {m_sarima["Bias"]:>8.2f}')

# ── Prediction intervals ─────────────────────────────────────────
print('\n=== Prediction Interval Evaluation ===')
# SARIMA confidence intervals
forecast_ci = sarima_fit.get_forecast(steps=h)
ci = np.exp(forecast_ci.conf_int(alpha=0.10))   # 90% interval
lower = ci.iloc[:,0].values
upper = ci.iloc[:,1].values

m_sarima_pi = forecast_metrics(test, pred_sarima, train, m=12,
                                 y_lower=lower, y_upper=upper)
print(f'SARIMA 90% PI coverage: {m_sarima_pi["Coverage%"]:.1f}% (target: 90%)')
interval_width = (upper - lower).mean()
print(f'Mean PI width: {interval_width:.1f} passengers')

# Diagnose coverage by horizon
print('\nCoverage by forecast horizon (should be ~90% at each step):')
for step in [1, 6, 12, 18, 24]:
    if step <= h:
        cov = ((test[step-1] >= lower[step-1]) and (test[step-1] <= upper[step-1]))
        print(f'  Step {step:>2}: actual={test[step-1]:.0f}  [{lower[step-1]:.0f}, {upper[step-1]:.0f}]  {'✓ INSIDE' if cov else '✗
OUTSIDE'}')

# ── Visualise metrics comparison ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Forecast Evaluation Metrics Comparison', fontsize=14, fontweight='bold')

methods = ['Seasonal Naive','Holt-Winters','SARIMA']
all_metrics = [m_naive, m_hw, m_sarima]

for ax, metric_name in zip(axes, ['MAE','MAPE%','MASE']):
    values = [m[metric_name] for m in all_metrics]
    colors = ['#BDBDBD','#1976D2','#43A047']
    bars = ax.bar(methods, values, color=colors, edgecolor='white')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(values)*0.01,
               f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'{metric_name} (lower=better)'); ax.grid(alpha=0.3, axis='y')
    ax.set_xticklabels(methods, rotation=30, ha='right')

plt.tight_layout()
plt.savefig('forecast_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
```